In [1]:
!pip install -q -U langchain langchain-community langchain-huggingface chromadb
!pip install -q -U sentence-transformers beautifulsoup4 transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.

In [ ]:
import os
from google.colab import userdata
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA

# 1. SET UP CREDENTIALS
# Add your HF token to Colab Secrets (left sidebar 🔑 icon) as 'HF_TOKEN'
# or uncomment the line below:
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = "your_token_here"
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HF_TOKEN')

# 2. LOAD DATA FROM ODU URLS
urls = [
    "https://catalog.odu.edu/graduate/data-science/#coursestext",
    "https://catalog.odu.edu/graduate/data-science/#programstext",
    "https://catalog.odu.edu/graduate/data-science/data-science-analytics-full-stack-ms/#curriculumtext"
]

loader = WebBaseLoader(urls)
data = loader.load()

# 3. SPLIT TEXT INTO CHUNKS
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(data)

# 4. CREATE EMBEDDINGS AND VECTOR STORE (Open Source Model)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)

# 5. INITIALIZE OPEN SOURCE LLM (Mistral-7B via Hugging Face Inference API)
repo_id = "mistralai/Mistral-7B-Instruct-v0.2"
llm = HuggingFaceEndpoint(
    repo_id=repo_id,
    max_new_tokens=512,
    temperature=0.1,
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"]
)

# 6. CREATE THE RAG CHAIN
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3})
)

def ask_odu(question):
    print(f"\nQuestion: {question}")
    response = qa_chain.invoke(question)
    print(f"Answer: {response['result']}")

# --- SAMPLE QUESTIONS ---
questions = [
    "What are the requirements for the Full Stack concentration?",
    "What are the core courses for the Data Science program?",
    "How many credit hours are required for the program?",
    "Does the program require a capstone project or a thesis?"
]

for q in questions:
    ask_odu(q)